In [63]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

In [64]:
df = pd.read_csv("../data/train.csv")
df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [65]:
df.describe()
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

In [66]:
df.drop_duplicates(inplace=True)
df.isnull().sum().sort_values(ascending=False)

PoolQC           1453
MiscFeature      1406
Alley            1369
Fence            1179
MasVnrType        872
                 ... 
MoSold              0
YrSold              0
SaleType            0
SaleCondition       0
SalePrice           0
Length: 81, dtype: int64

In [67]:
# ==========================================================
# Handling Missing Values
# ==========================================================

# Fill categorical features where NaN means the feature
# does not exist rather than missing information.

fill_values = {
    "PoolQC": "NoPool",
    "MiscFeature": "NoMiscFeature",
    "Alley": "NoAlley",
    "Fence": "NoFence",
    "MasVnrType": "NoMasonry",
    "FireplaceQu": "NoFireplace",
    "GarageType": "NoGarage",
    "GarageFinish": "NoGarage",
    "GarageQual": "NoGarage",
    "GarageCond": "NoGarage",
    "BsmtQual": "NoBasement",
    "BsmtCond": "NoBasement",
    "BsmtExposure": "NoBasement",
    "BsmtFinType1": "NoBasement",
    "BsmtFinType2": "NoBasement"
}

for col, value in fill_values.items():
    df[col] = df[col].fillna(value)

# ==========================================================
# Maintain Garage Consistency
# ==========================================================

garage_cat = [
    "GarageType",
    "GarageFinish",
    "GarageQual",
    "GarageCond"
]

for col in garage_cat:
    df.loc[df["GarageType"] == "NoGarage", col] = "NoGarage"

# ==========================================================
# LotFrontage Imputation
# ==========================================================

df["LotFrontage"] = (
    df.groupby("Neighborhood")["LotFrontage"]
    .transform(lambda x: x.fillna(x.median()))
)

# In case an entire neighborhood has missing values
df["LotFrontage"] = df["LotFrontage"].fillna(df["LotFrontage"].median())

# ==========================================================
# Masonry Veneer Area
# ==========================================================

df.loc[df["MasVnrType"] == "NoMasonry", "MasVnrArea"] = 0
df["MasVnrArea"] = df["MasVnrArea"].fillna(0)

# ==========================================================
# Electrical System
# ==========================================================

df["Electrical"] = (
    df.groupby("Neighborhood")["Electrical"]
    .transform(lambda x: x.fillna(x.mode()[0]))
)

# Fallback if any NaNs remain
df["Electrical"] = df["Electrical"].fillna(df["Electrical"].mode()[0])

# ==========================================================
# Maintain Garage Consistency (Numerical)
# ==========================================================

garage_num = [
    "GarageYrBlt",
    "GarageCars",
    "GarageArea"
]

for col in garage_num:
    df.loc[df["GarageType"] == "NoGarage", col] = 0

# ==========================================================
# Final Check
# ==========================================================

print(df.isna().sum()[df.isna().sum() > 0])

Series([], dtype: int64)


In [68]:
# =========================
# Feature Engineering
# =========================

# Total house area
df["TotalSF"] = (
        df["TotalBsmtSF"] +
        df["1stFlrSF"] +
        df["2ndFlrSF"]
)

# House age
df["HouseAge"] = df["YrSold"] - df["YearBuilt"]

# Years since last remodel
df["RemodelAge"] = df["YrSold"] - df["YearRemodAdd"]

# Garage age
df["GarageAge"] = np.where(
    df["GarageYrBlt"] == 0,
    0,
    df["YrSold"] - df["GarageYrBlt"]
)

# Total bathrooms
df["TotalBathrooms"] = (
        df["FullBath"] +
        0.5 * df["HalfBath"] +
        df["BsmtFullBath"] +
        0.5 * df["BsmtHalfBath"]
)

# Total porch area
df["TotalPorchSF"] = (
        df["OpenPorchSF"] +
        df["EnclosedPorch"] +
        df["3SsnPorch"] +
        df["ScreenPorch"]
)

# Total outdoor area
df["TotalOutdoorSF"] = (
        df["WoodDeckSF"] +
        df["TotalPorchSF"]
)

# Binary features
df["HasGarage"] = (df["GarageArea"] > 0).astype(int)
df["HasBasement"] = (df["TotalBsmtSF"] > 0).astype(int)
df["HasFireplace"] = (df["Fireplaces"] > 0).astype(int)
df["HasPool"] = (df["PoolArea"] > 0).astype(int)
df["Has2ndFloor"] = (df["2ndFlrSF"] > 0).astype(int)
df["HasRemodel"] = (df["YearBuilt"] != df["YearRemodAdd"]).astype(int)

# Optional
df["LotAreaPerRoom"] = df["LotArea"] / df["TotRmsAbvGrd"]

In [69]:
# TO Check if all the columns are cleaned and NAN is filled
df.isnull().sum()[df.isnull().sum() > 0]

df.to_csv("../data/Cleaned_Housing_Dataset.csv" , sep = ',')


In [70]:
df.info()

cat_col = df.select_dtypes(include='object').columns
numerical_col = df.select_dtypes(exclude='object').columns

ordinal_cols = [
    "LotShape",
    "LandSlope",
    "ExterQual",
    "ExterCond",
    "BsmtQual",
    "BsmtCond",
    "BsmtExposure",
    "BsmtFinType1",
    "BsmtFinType2",
    "HeatingQC",
    "KitchenQual",
    "Functional",
    "FireplaceQu",
    "GarageFinish",
    "GarageQual",
    "GarageCond",
    "PoolQC",
    "Fence",
    "OverallQual",
    "OverallCond"
]
nominal_cols = [
    "MSSubClass",
    "MSZoning",
    "Street",
    "Alley",
    "Neighborhood",
    "Condition1",
    "Condition2",
    "BldgType",
    "HouseStyle",
    "RoofStyle",
    "RoofMatl",
    "Exterior1st",
    "Exterior2nd",
    "MasVnrType",
    "Foundation",
    "Heating",
    "CentralAir",
    "Electrical",
    "GarageType",
    "MiscFeature",
    "SaleType",
    "SaleCondition"
]




<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 95 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Id              1460 non-null   int64  
 1   MSSubClass      1460 non-null   int64  
 2   MSZoning        1460 non-null   object 
 3   LotFrontage     1460 non-null   float64
 4   LotArea         1460 non-null   int64  
 5   Street          1460 non-null   object 
 6   Alley           1460 non-null   object 
 7   LotShape        1460 non-null   object 
 8   LandContour     1460 non-null   object 
 9   Utilities       1460 non-null   object 
 10  LotConfig       1460 non-null   object 
 11  LandSlope       1460 non-null   object 
 12  Neighborhood    1460 non-null   object 
 13  Condition1      1460 non-null   object 
 14  Condition2      1460 non-null   object 
 15  BldgType        1460 non-null   object 
 16  HouseStyle      1460 non-null   object 
 17  OverallQual     1460 non-null   i

In [71]:
df["MSSubClass"] = df["MSSubClass"].astype("object")

In [72]:
# ==========================================================
# Ordinal Encoding
# ==========================================================

quality_map = {
    "None": 0,
    "Po": 1,
    "Fa": 2,
    "TA": 3,
    "Gd": 4,
    "Ex": 5
}

lotshape_map = {
    "Reg": 0,
    "IR1": 1,
    "IR2": 2,
    "IR3": 3
}

landslope_map = {
    "Gtl": 0,
    "Mod": 1,
    "Sev": 2
}

bsmtexposure_map = {
    "None": 0,
    "No": 1,
    "Mn": 2,
    "Av": 3,
    "Gd": 4
}

bsmtfin_map = {
    "None": 0,
    "Unf": 1,
    "LwQ": 2,
    "Rec": 3,
    "BLQ": 4,
    "ALQ": 5,
    "GLQ": 6
}

garagefinish_map = {
    "None": 0,
    "Unf": 1,
    "RFn": 2,
    "Fin": 3
}

fence_map = {
    "None": 0,
    "MnWw": 1,
    "GdWo": 2,
    "MnPrv": 3,
    "GdPrv": 4
}

functional_map = {
    "Sal": 0,
    "Sev": 1,
    "Maj2": 2,
    "Maj1": 3,
    "Mod": 4,
    "Min2": 5,
    "Min1": 6,
    "Typ": 7
}

# Apply mappings

quality_cols = [
    "ExterQual",
    "ExterCond",
    "BsmtQual",
    "BsmtCond",
    "HeatingQC",
    "KitchenQual",
    "FireplaceQu",
    "GarageQual",
    "GarageCond",
    "PoolQC"
]

for col in quality_cols:
    df[col] = df[col].map(quality_map)

df["LotShape"] = df["LotShape"].map(lotshape_map)

df["LandSlope"] = df["LandSlope"].map(landslope_map)

df["BsmtExposure"] = df["BsmtExposure"].map(bsmtexposure_map)

df["BsmtFinType1"] = df["BsmtFinType1"].map(bsmtfin_map)
df["BsmtFinType2"] = df["BsmtFinType2"].map(bsmtfin_map)

df["GarageFinish"] = df["GarageFinish"].map(garagefinish_map)

df["Fence"] = df["Fence"].map(fence_map)

df["Functional"] = df["Functional"].map(functional_map)

# OverallQual and OverallCond are already ordinal (1-10)
# No encoding required.


# ==========================================================
# One-Hot Encoding (Nominal Features)
# ==========================================================

nominal_cols = [
    "MSSubClass",
    "MSZoning",
    "Street",
    "Alley",
    "LandContour",
    "Utilities",
    "LotConfig",
    "Neighborhood",
    "Condition1",
    "Condition2",
    "BldgType",
    "HouseStyle",
    "RoofStyle",
    "RoofMatl",
    "Exterior1st",
    "Exterior2nd",
    "MasVnrType",
    "Foundation",
    "Heating",
    "CentralAir",
    "Electrical",
    "GarageType",
    "MiscFeature",
    "SaleType",
    "SaleCondition"
]

ohe = OneHotEncoder(
    sparse_output=False,
    drop = "first",
    handle_unknown="ignore"
)

encoded = ohe.fit_transform(df[nominal_cols])

encoded_df = pd.DataFrame(
    encoded,
    columns=ohe.get_feature_names_out(nominal_cols),
    index = df.index
)

df = df.drop(columns=nominal_cols)
df = pd.concat([df , encoded_df] , axis = 1)

In [73]:
df.shape

(1460, 231)

In [74]:
df.to_csv("../data/Encoded_Cleaned_Housing_Dataset.csv")